In [1]:
import os
from style import *
import requests

import pandas as pd

In [2]:
DATA_DIR = "MyData"
df = pd.read_csv(os.path.join(DATA_DIR, "final.csv"))

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38425 entries, 0 to 38424
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Time               38425 non-null  object 
 1   Platform           38425 non-null  object 
 2   ms Played          38425 non-null  int64  
 3   Country            38425 non-null  object 
 4   IP Address         38425 non-null  object 
 5   Track Name         38425 non-null  object 
 6   Artist Name        38425 non-null  object 
 7   Album Name         38425 non-null  object 
 8   Spotify URI        38425 non-null  object 
 9   Reason Start       38425 non-null  object 
 10  Reason End         38425 non-null  object 
 11  Shuffle            38425 non-null  bool   
 12  Offline            38425 non-null  float64
 13  Offline Timestamp  38425 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(11)
memory usage: 3.8+ MB


In [4]:
ips = df["IP Address"].unique()
ips

array(['47.8.102.145', '47.8.113.175', '47.8.96.228', ...,
       '47.31.136.234', '47.31.142.60', '47.31.149.252'], dtype=object)

In [5]:
df["IP_Two"] = df["IP Address"].apply(lambda x: ".".join(x.split(".")[:2]))

In [74]:
two_uniques = df["IP_Two"].unique()
two_uniques

array(['47.8', '117.211', '103.55', '47.9', '59.144', '139.59', '192.241',
       '117.220', '206.189', '47.15', '165.232', '188.166', '169.197',
       '157.230', '162.221', '14.139', '112.79', '128.14', '37.19',
       '114.31', '79.110', '138.199', '91.245', '91.219', '172.96',
       '45.249', '203.81', '103.217', '202.142', '194.5', '23.251',
       '8.17', '74.208', '51.81', '146.70', '132.154', '139.167', '47.31',
       '103.27', '143.110', '137.184', '157.37', '162.243', '217.148',
       '107.155', '42.105', '112.133'], dtype=object)

In [62]:
ips_to_find = []
found_ip_twos = set()
for ip in ips:
    first_two = ".".join(ip.split(".")[:2])
    if first_two not in found_ip_twos:
        ips_to_find.append(ip)
    found_ip_twos.add(first_two)

In [64]:
len(ips_to_find) == len(two_uniques)

True

In [82]:
res = requests.get(f'https://ipapi.co/47.8.102.145/json/')
res = res.json()
res

{'ip': '47.8.102.145',
 'network': '47.8.102.0/24',
 'version': 'IPv4',
 'city': 'Mumbai',
 'region': 'Maharashtra',
 'region_code': 'MH',
 'country': 'IN',
 'country_name': 'India',
 'country_code': 'IN',
 'country_code_iso3': 'IND',
 'country_capital': 'New Delhi',
 'country_tld': '.in',
 'continent_code': 'AS',
 'in_eu': False,
 'postal': '400070',
 'latitude': 19.0748,
 'longitude': 72.8856,
 'timezone': 'Asia/Kolkata',
 'utc_offset': '+0530',
 'country_calling_code': '+91',
 'currency': 'INR',
 'currency_name': 'Rupee',
 'languages': 'en-IN,hi,bn,te,mr,ta,ur,gu,kn,ml,or,pa,as,bh,sat,ks,ne,sd,kok,doi,mni,sit,sa,fr,lus,inc',
 'country_area': 3287590.0,
 'country_population': 1352617328,
 'asn': 'AS55836',
 'org': 'Reliance Jio Infocomm Limited'}

In [83]:
def get_ip_info(ip_address):
    res = requests.get(f'https://ipapi.co/{ip_address}/json/')
    first_two = ".".join(ip.split(".")[:2])
    res = res.json()
    city = res["city"]
    region = res["region"]
    region_code = res["region_code"]
    country = res["country_name"]
    country_code = res["country_code_iso3"]
    postal = res["postal"]
    lattitude = res["latitude"]
    longitude = res["longitude"]
    org = res["org"]
    info = {
        "City": city,
        "Region": region,
        "Region Code": region_code,
        "Country": country,
        "Country Code": country_code,
        "Postal Code": postal,
        "Lattitude": lattitude,
        "Longitude": longitude,
        "Service Provider": org,
        "IP Two": first_two
    }
    return info

In [84]:
infos = []
i=1
for ip in ips_to_find:
    cprint(f"{i}/{len(ips_to_find)}", "green", end="\r")
    info = get_ip_info(ip)
    infos.append(info)
    i+=1

In [85]:
infos[0]

{'City': 'Mumbai',
 'Region': 'Maharashtra',
 'Region Code': 'MH',
 'Country': 'India',
 'Country Code': 'IND',
 'Postal Code': '400070',
 'Lattitude': 19.0748,
 'Longitude': 72.8856,
 'Service Provider': 'Reliance Jio Infocomm Limited',
 'IP Two': '47.8'}

In [86]:
ip_df = pd.DataFrame(infos)

In [88]:
ip_df.to_csv(os.path.join(DATA_DIR, "IP_Info.csv"), index=False)